In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kandeelai22/messy-e-commerce-sales-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/kriti/.cache/kagglehub/datasets/kandeelai22/messy-e-commerce-sales-dataset/versions/1


### Import Libraries

In [3]:
import numpy as np
import pandas as pd

### Loading and Exploring Dataset

In [4]:
raw_df = pd.read_csv("messy_ecommerce_sales_data.csv")

In [5]:
raw_df.shape

(103, 11)

In [6]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1    Customer_Name  103 non-null    str    
 2   Order_ID        103 non-null    str    
 3   Order_Date      103 non-null    str    
 4   Product         103 non-null    str    
 5    Category       95 non-null     str    
 6   Quantity        98 non-null     str    
 7   Price           98 non-null     str    
 8   Payment_Method  103 non-null    str    
 9   Status          103 non-null    str    
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), str(9)
memory usage: 16.1 KB


#### Missing values

In [7]:
missing = raw_df.isnull().sum()
missing_percentage = (missing / len(raw_df) * 100).round(2)
audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_percentage})
audit[audit['Missing %'] > 0]

,Missing Count,Missing %
Category,8,7.77
Quantity,5,4.85
Price,5,4.85
Total,14,13.59


#### Unique Values per Column

In [8]:
for col in raw_df.columns:
    print(f'Column {col}: {raw_df[col].nunique()}')

Column ID: 100
Column  Customer_Name: 100
Column Order_ID: 100
Column Order_Date: 86
Column Product: 21
Column  Category: 9
Column Quantity: 8
Column Price: 94
Column Payment_Method: 4
Column Status: 5
Column Total: 88


#### Outliers

In [9]:
data_iqr = raw_df['Total']

iqr = data_iqr.quantile(0.75) - data_iqr.quantile(0.25)
lower_bound = data_iqr.quantile(0.25) - 1.5 * iqr
upper_bound = data_iqr.quantile(0.75) + 1.5 * iqr

data_iqr[(data_iqr > upper_bound) | (data_iqr < lower_bound)].shape[0]

3

#### Stripping column names

In [10]:
raw_df.columns = raw_df.columns.str.strip()


In [11]:
raw_df['Category'].str.strip().str.title().value_counts()

Category
Books          22
Home           20
Electronics    17
Sports         17
Clothing       15
Electronic      4
Name: count, dtype: int64

### Data Cleaning

#### Date inconsistencies

In [12]:
raw_df['Order_Date'].unique()[:15]


<ArrowStringArray>
['11/22/2024',   '7/5/2025', '12/23/2024',  '3/19/2025', '10/20/2025',
 '11/20/2024',   '2/2/2025',   '1/3/2025', '10/23/2025',   '5/3/2025',
  '9/26/2025', '10/22/2025',   '3/2/2025',  '7/24/2025', 'Jan 5 2023']
Length: 15, dtype: str

In [13]:
raw_df['Order_Date'] = pd.to_datetime(raw_df['Order_Date'], errors='coerce')
raw_df = raw_df[raw_df['Order_Date'].notna()]

In [45]:
raw_df['Year'] = raw_df['Order_Date'].dt.year
raw_df['Month'] = raw_df['Order_Date'].dt.month
raw_df['Day'] = raw_df['Order_Date'].dt.day
raw_df['DayOfWeek'] = raw_df['Order_Date'].dt.dayofweek


In [14]:
raw_df['Order_Date'].unique()[:15]

<DatetimeArray>
['2024-11-22 00:00:00', '2025-07-05 00:00:00', '2024-12-23 00:00:00',
 '2025-03-19 00:00:00', '2025-10-20 00:00:00', '2024-11-20 00:00:00',
 '2025-02-02 00:00:00', '2025-01-03 00:00:00', '2025-10-23 00:00:00',
 '2025-05-03 00:00:00', '2025-09-26 00:00:00', '2025-10-22 00:00:00',
 '2025-03-02 00:00:00', '2025-07-24 00:00:00', '2025-01-10 00:00:00']
Length: 15, dtype: datetime64[us]

In [15]:
raw_df[raw_df['Order_Date'].isna()]

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total


In [16]:
mode_date = raw_df['Order_Date'].mode()[0]
raw_df['Order_Date'] = raw_df['Order_Date'].fillna(mode_date)

In [17]:
raw_df['Order_Date'].isna().sum()

np.int64(0)

#### All duplicate rows and their first occurence

In [18]:
raw_df[raw_df.duplicated(subset=['Order_ID'], keep=False)]

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
42,142,Customer_142,ORD-69018,2025-10-30,Shoes,Clothing,5,645.26,Credit Card,Shipped,2258.410
46,146,Customer_146,ORD-32755,2025-07-09,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.840
75,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,1,111.36,Credit Card,Processing,111.360
100,175,Customer_175,ORD-56651,2025-02-24,Headphones,Electronics,1,111.36,Credit Card,Processing,77.952
101,142,Customer_142,ORD-69018,2025-10-30,Shoes,Clothing,5,645.26,Credit Card,Shipped,3226.300
102,146,Customer_146,ORD-32755,2025-07-09,Basketball,electronic,2,705.42,Bank Transfer,Processing,1410.840


In [19]:
raw_df['Order_ID'].value_counts()[raw_df['Order_ID'].value_counts() > 1]

Order_ID
ORD-69018    2
ORD-32755    2
ORD-56651    2
Name: count, dtype: int64

In [20]:
raw_df = raw_df.drop_duplicates(subset=['Order_ID'], keep = 'first')

In [21]:
raw_df['Order_ID'].value_counts()[raw_df['Order_ID'].value_counts() > 1]

Series([], Name: count, dtype: int64)

#### Standardize Category

In [22]:
raw_df['Category'].value_counts()

Category
Books          22
Home           20
Sports         16
Clothing       13
Electronics     9
electronic      3
ELECTRONICS     3
electronics     3
sports          1
Name: count, dtype: int64

In [23]:
raw_df['Category'] = raw_df['Category'].str.strip().str.title()

In [24]:
raw_df['Category'].value_counts()

Category
Books          22
Home           20
Sports         17
Electronics    15
Clothing       13
Electronic      3
Name: count, dtype: int64

In [25]:
raw_df[raw_df['Category'].isna()]

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
33,133,Customer_133,ORD-68182,2024-12-05,Biography,NaN,5,343.24,Credit Card,Shipped,1716.20
36,136,Customer_136,ORD-20985,2025-06-12,Headphones,NaN,1,696.71,Credit Card,Delivered,696.71
80,180,Customer_180,ORD-86629,2025-03-26,Laptop,NaN,2,418.38,Credit Card,Processing,836.76
81,181,Customer_181,ORD-54481,2024-12-29,Smartphone,NaN,1,266.48,Cash on Delivery,Cancelled,266.48
82,182,Customer_182,ORD-79672,2025-08-03,Shoes,NaN,4,538.35,PayPal,Processing,2153.40
84,184,Customer_184,ORD-67799,2025-02-15,Jeans,NaN,4,101.19,Bank Transfer,Processing,404.76
93,193,Customer_193,ORD-42475,2025-06-06,Basketball,NaN,NaN,522.02,PayPal,Shipped,NaN
98,198,Customer_198,ORD-14608,2025-07-27,Vacuum,NaN,2,497.01,Cash on Delivery,Shipped,994.02


In [26]:
category_map = {
    'Sports': 'Sports',
    'Electronic': 'Electronics',
    'Electronics': 'Electronics',
    'NaN': np.nan
}

In [27]:
raw_df['Category'] = raw_df['Category'].replace(category_map)

In [28]:
raw_df['Category'].value_counts()

Category
Books          22
Home           20
Electronics    18
Sports         17
Clothing       13
Name: count, dtype: int64

In [29]:
# fill missing category
raw_df['Category'] = raw_df['Category'].fillna('Unknown')

#### Clean Price Column

In [30]:
raw_df['Price']

0         38
1        abd
2     389.05
3     233.92
4     552.51
       ...  
95    817.46
96       abd
97    160.16
98    497.01
99    372.28
Name: Price, Length: 98, dtype: str

In [31]:
raw_df['Price'] = raw_df['Price'].astype(str).replace(r'[$,]', '', regex = True).str.strip()

In [32]:
raw_df['Price'].sample(20)

20       300
2     389.05
69    582.13
16       NaN
68    141.49
75    111.36
29    736.71
57     275.1
35    446.19
42    645.26
65    443.13
78    472.14
53     278.9
72    942.76
0         38
86    365.34
59    163.16
70    741.63
37      -100
62    155.76
Name: Price, dtype: str

In [33]:
raw_df['Price'] = pd.to_numeric(raw_df['Price'], errors = 'coerce')

In [34]:
# remove any negative price
raw_df = raw_df[~(raw_df['Price'] < 0)]

In [35]:
raw_df['Price'].dtype

dtype('float64')

#### Clean Quantity Column

In [36]:
raw_df['Quantity'].sample(20)

51      2
76      5
62      2
52      3
94      5
55      1
68      2
57      4
38      1
69      1
96    NaN
90      1
5       3
42      5
77      1
16      4
7       5
36      1
8       1
81      1
Name: Quantity, dtype: str

In [37]:
raw_df['Quantity'] = pd.to_numeric(raw_df['Quantity'].astype(str).str.extract(r'(-?\d+)', expand=False), errors='coerce')

In [38]:
raw_df['Quantity'].sample(20)

76    5.0
29    4.0
36    1.0
83    5.0
52    3.0
41    3.0
18    5.0
82    4.0
60    5.0
54    5.0
15    2.0
62    2.0
39    2.0
96    NaN
61    4.0
98    2.0
88    5.0
93    NaN
24    5.0
70    5.0
Name: Quantity, dtype: float64

In [39]:
# remove any negative quantity
raw_df = raw_df[~(raw_df['Quantity'] < 0)]

In [40]:
raw_df['Quantity'].sample(20)

44    5.0
94    5.0
19    4.0
91    2.0
96    NaN
9     5.0
22    2.0
21    4.0
43    5.0
54    5.0
89    1.0
66    2.0
4     1.0
63    3.0
51    2.0
11    2.0
13    1.0
82    4.0
88    5.0
90    1.0
Name: Quantity, dtype: float64

#### Clean Total Column

In [41]:
raw_df['Total'].sample(20)

68     282.980
41    1638.120
95    4087.300
51     640.024
28    2696.070
45    1721.980
85    1023.288
77      69.480
84     404.760
98     994.020
6          NaN
1          NaN
56         NaN
66    1138.760
93         NaN
22    1113.860
67         NaN
91    1762.040
38     531.060
42    2258.410
Name: Total, dtype: float64

In [42]:
raw_df['Total'].isna().sum()

np.int64(13)

In [43]:
def parse_total(value):
    # check if value is string
    if isinstance(value, str):
        cleaned = value.replace('$', '').replace(',','')
        try:
            return float(cleaned)
        except ValueError:
            return np.nan 
    # check if value is int or float
    elif isinstance(value, (int, float)):
        return value
    else:
        return np.nan

raw_df['Total'] = raw_df['Total'].apply(parse_total)

In [44]:
raw_df['Total'].isna().sum()

np.int64(13)

In [53]:
raw_df['Total'].sample(20)

5      366.180
62     311.520
87    2648.880
74     749.920
44    4245.350
26    3347.200
82    2153.400
79    2606.010
93         NaN
0      114.000
51     640.024
36     696.710
52     699.330
70    3708.150
4      552.510
41    1638.120
94    4722.700
28    2696.070
91    1762.040
21    2092.040
Name: Total, dtype: float64